# RAG-Based Semester Course Advisor
**Team project – KBA assignment** · Retrieval-Augmented Generation (RAG) system that recommends the courses a student should register for next semester.

**Pipeline in one picture**

```
KB files (catalog, rules, degree plan) ─► parse ─► chunk ─► embed ─► vector index
                                                                         │
Student profile ─► query ─► retrieve top-k passages ─────────────────────┤
       │                                                                 ▼
       └──────► Python rule engine (R1–R14) ──► eligible / flagged / blocked ─► ranking + schedule builder
                                                                         │
                     LLM (NVIDIA NIM free API, or offline template) writes grounded reasons ◄─┘
                                                                         ▼
                      Recommendation + sources + exclusions + audit  ─►  Chatbot
```

> ⚠️ **All data is fictional.** The "Nile Valley University" catalog and rules were written for this assignment (no official documents were used).

### How to run
1. Keep the folders `kb/` and `data/` next to this notebook (`outputs/` is created automatically).
2. `pip install -r requirements.txt` (only `numpy` and `scikit-learn` are mandatory; `sentence-transformers` is optional and gives better embeddings).
3. *(Optional but recommended)* set `NVIDIA_API_KEY` from https://build.nvidia.com/settings so the free NVIDIA NIM LLM writes explanations and powers the chatbot. You can also use `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`. Without a key the notebook uses a deterministic, evidence-grounded template so everything still runs offline.
4. **Run all cells.** Section 8 runs the 6 test profiles; Section 9 demonstrates the chatbot.


## 1. Setup and configuration

In [1]:
import os, re, json, textwrap, urllib.request, urllib.error
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional
import numpy as np

KB_DIR   = Path("kb")
DATA_DIR = Path("data")
OUT_DIR  = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

EMBEDDING_BACKEND = "auto"   # "auto" -> sentence-transformers if installed, else TF-IDF | "sbert" | "tfidf"
TOP_K_COURSES     = 12       # how many retrieved course passages are inspected for the "excluded" list
SBERT_MODEL       = "sentence-transformers/all-MiniLM-L6-v2"

GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D+", "D", "F"]   # best -> worst
def grade_ok(grade, minimum):            # True if `grade` is at least as good as `minimum`
    return GRADES.index(grade) <= GRADES.index(minimum)

print("Knowledge-base files:", sorted(p.name for p in KB_DIR.glob("*.md")))

Knowledge-base files: ['course_catalog.md', 'degree_plan.md', 'registration_rules.md']


## 2. Knowledge base: parsing and chunking
The three Markdown documents are parsed into **structured records** (used by the Python rule engine) *and* into **text chunks** (used for embedding and retrieval).
Rule parameters (e.g. `max_credits=18`) are read from the rules file, so changing the policy file changes the system's behaviour.

In [2]:
@dataclass
class Meeting:
    days: list; start: int; end: int
    def overlaps(self, other):
        return bool(set(self.days) & set(other.days)) and self.start < other.end and other.start < self.end
    def __str__(self):
        f = lambda m: f"{m//60:02d}:{m%60:02d}"
        return f"{'/'.join(self.days)} {f(self.start)}-{f(self.end)}"

@dataclass
class Course:
    code: str; title: str; credits: Optional[int]; level: Optional[int]
    offered: Optional[list]; prereqs: Optional[list]; coreqs: list
    min_earned: Optional[int]; schedule: Optional[Meeting]
    category: str; core_year: Optional[int]; workload: str
    description: str; topics: list; missing: list
    @property
    def source(self): return f"course_catalog.md#{self.code}"

@dataclass
class Rule:
    rid: str; title: str; params: dict; text: str
    @property
    def source(self): return f"registration_rules.md#{self.rid}"

@dataclass
class Chunk:
    cid: str; kind: str; source: str; title: str; text: str

UNPUBLISHED = {"", "tba", "not published", "not yet announced", "n/a"}

def _field_list(v):
    return [] if v.strip().lower() == "none" else [x.strip() for x in v.split(",") if x.strip()]

def _parse_meeting(s):
    m = re.match(r"([A-Za-z/]+)\s+(\d{2}):(\d{2})-(\d{2}):(\d{2})", s.strip())
    if not m: return None
    days = m.group(1).split("/")
    return Meeting(days, int(m.group(2))*60+int(m.group(3)), int(m.group(4))*60+int(m.group(5)))

def parse_catalog(path):
    courses = {}
    for block in re.split(r"\n(?=## )", Path(path).read_text(encoding="utf-8")):
        h = re.match(r"## ([A-Z]{2,5}\d{3}) — (.+)", block)
        if not h: continue
        code, title = h.groups()
        f = dict(re.findall(r"^- ([A-Za-z ]+): (.*)$", block, flags=re.M))
        missing = []
        def pub(name):                       # returns raw value or None when not published
            v = f.get(name, "").strip()
            if v.lower() in UNPUBLISHED:
                missing.append(name.lower()); return None
            return v
        offered = pub("Offered"); pre = pub("Prerequisites"); sched = pub("Schedule"); cr = pub("Credits"); lv = pub("Level")
        cat = f.get("Category", "").strip(); cy = re.search(r"\(Year (\d)\)", cat)
        me = f.get("Min credits earned", "None").strip()
        courses[code] = Course(
            code=code, title=title.strip(),
            credits=int(cr) if cr else None, level=int(lv) if lv else None,
            offered=[x.strip() for x in offered.split(",")] if offered else None,
            prereqs=_field_list(pre) if pre is not None else None,
            coreqs=_field_list(f.get("Corequisites", "None")),
            min_earned=int(me) if me.isdigit() else None,
            schedule=_parse_meeting(sched) if sched else None,
            category=cat, core_year=int(cy.group(1)) if cy else None,
            workload=f.get("Workload", "Medium").strip(),
            description=f.get("Description", "").strip(),
            topics=[t.strip() for t in f.get("Topics", "").split(";") if t.strip()],
            missing=missing)
    return courses

def parse_rules(path):
    rules = {}
    for block in re.split(r"\n(?=## )", Path(path).read_text(encoding="utf-8")):
        h = re.match(r"## (R\d+) — (.+)", block)
        if not h: continue
        rid, title = h.groups()
        ptxt = re.search(r"- Parameters: (.*)", block).group(1)
        params = {}
        for kv in ptxt.split(";"):
            if "=" in kv:
                k, v = kv.strip().split("=", 1)
                params[k] = float(v) if re.fullmatch(r"[\d.]+", v) else v
        rules[rid] = Rule(rid, title.strip(), params, re.search(r"- Text: (.*)", block).group(1).strip())
    return rules

def split_text(text, max_chars=900, overlap=120):
    """Sentence-aware splitter: long passages are cut into overlapping chunks (short ones stay whole)."""
    if len(text) <= max_chars: return [text]
    sents, chunks, cur = re.split(r"(?<=[.!?])\s+", text), [], ""
    for s in sents:
        if len(cur) + len(s) > max_chars and cur:
            chunks.append(cur.strip()); cur = cur[-overlap:] + " "
        cur += s + " "
    if cur.strip(): chunks.append(cur.strip())
    return chunks

def build_chunks(courses, rules, plan_path):
    chunks = []
    for c in courses.values():
        pre = "not published" if c.prereqs is None else (", ".join(c.prereqs) or "none")
        text = (f"{c.code} {c.title}. Level {c.level}. {c.credits} credits. Offered: {', '.join(c.offered) if c.offered else 'not published'}. "
                f"Prerequisites: {pre}. Category: {c.category}. Workload: {c.workload}. {c.description} Topics: {'; '.join(c.topics)}.")
        for i, part in enumerate(split_text(text)):
            chunks.append(Chunk(f"{c.code}" + (f"-{i}" if i else ""), "course", c.source, f"{c.code} {c.title}", part))
    for r in rules.values():
        chunks.append(Chunk(r.rid, "rule", r.source, f"{r.rid} {r.title}", f"{r.rid} {r.title}. {r.text}"))
    plan = Path(plan_path).read_text(encoding="utf-8")
    for sec in re.split(r"\n(?=### )", plan):
        h = re.match(r"### (.+)", sec)
        if not h: continue
        body = sec.split("\n", 1)[1].strip()
        for i, part in enumerate(split_text(body)):
            chunks.append(Chunk(f"PLAN:{h.group(1)}" + (f"-{i}" if i else ""), "plan", f"degree_plan.md#{h.group(1)}", h.group(1), f"{h.group(1)}. {part}"))
    return chunks

COURSES = parse_catalog(KB_DIR / "course_catalog.md")
RULES   = parse_rules(KB_DIR / "registration_rules.md")
CHUNKS  = build_chunks(COURSES, RULES, KB_DIR / "degree_plan.md")

print(f"Parsed {len(COURSES)} courses, {len(RULES)} rules -> {len(CHUNKS)} chunks "
      f"({sum(c.kind=='course' for c in CHUNKS)} course / {sum(c.kind=='rule' for c in CHUNKS)} rule / {sum(c.kind=='plan' for c in CHUNKS)} degree-plan)")
print("Example chunk:\n ", CHUNKS[0].source, "→", textwrap.shorten(CHUNKS[0].text, 220))
print("Courses with incomplete catalog records:", [c.code for c in COURSES.values() if c.missing])

Parsed 32 courses, 14 rules -> 54 chunks (32 course / 14 rule / 8 degree-plan)
Example chunk:
  course_catalog.md#CS101 → CS101 Introduction to Programming. Level 100. 3 credits. Offered: Fall, Spring. Prerequisites: none. Category: Core (Year 1). Workload: Medium. First programming course using Python. Students learn to break [...]
Courses with incomplete catalog records: ['CS499']


## 3. Embeddings and vector index
`sentence-transformers` (all-MiniLM-L6-v2) is used when installed; otherwise the system falls back to a TF-IDF (unigram+bigram) vector space from scikit-learn, which needs no download. Retrieval is cosine similarity over L2-normalised vectors.

In [3]:
class VectorIndex:
    def __init__(self, chunks, backend="auto"):
        self.chunks, self.model, self.tfidf = chunks, None, None
        texts = [c.text for c in chunks]
        if backend in ("auto", "sbert"):
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer(SBERT_MODEL)
                self.name = f"sentence-transformers ({SBERT_MODEL.split('/')[-1]})"
                self.matrix = np.asarray(self.model.encode(texts, normalize_embeddings=True))
            except Exception as e:
                if backend == "sbert": raise
                self.model = None
        if self.model is None:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
            mat = self.tfidf.fit_transform(texts).toarray().astype(float)
            # L2-normalise rows so matrix @ query is cosine similarity (matches SBERT path)
            norms = np.linalg.norm(mat, axis=1, keepdims=True); norms[norms == 0] = 1.0
            self.matrix = mat / norms
            self.name = "TF-IDF unigram+bigram (scikit-learn fallback)"
        self.kinds = np.array([c.kind for c in chunks])

    def _encode(self, q):
        if self.model is not None:
            return np.asarray(self.model.encode([q], normalize_embeddings=True))[0]
        vec = self.tfidf.transform([q]).toarray()[0].astype(float)
        n = np.linalg.norm(vec)
        return vec / n if n else vec

    def search(self, query, k=5, kind=None):
        """Return [(chunk, cosine_score)] best-first, optionally restricted to one kind of chunk."""
        sims = self.matrix @ self._encode(query)
        order = np.argsort(-sims)
        out = [(self.chunks[i], float(sims[i])) for i in order if kind is None or self.kinds[i] == kind]
        return out[:k]

INDEX = VectorIndex(CHUNKS, EMBEDDING_BACKEND)
print("Embedding backend:", INDEX.name, "| matrix shape:", INDEX.matrix.shape)
for ch, s in INDEX.search("which courses teach neural networks and deep learning", k=3):
    print(f"  {s:.3f}  {ch.source}  [{ch.kind}]")


Embedding backend: TF-IDF unigram+bigram (scikit-learn fallback) | matrix shape: (54, 1864)
  0.351  course_catalog.md#AI401  [course]
  0.175  course_catalog.md#CS370  [course]
  0.168  degree_plan.md#Artificial Intelligence track  [plan]


## 4. Student profile
The profile holds level, semester, GPA, completed courses with grades, skills, interests/goals, desired credit load and optional requests. A missing GPA (e.g. a new student) is **not invented**: it is flagged and the GPA-dependent rules (R2, R3, R13) are reported as "not checkable".

In [4]:
@dataclass
class StudentProfile:
    student_id: str; name: str; academic_year: int; semester: str
    gpa: Optional[float]; completed: dict; desired_credits: int
    other_credits: int = 0
    skills: list = field(default_factory=list); interests: list = field(default_factory=list)
    goals: str = ""; workload_pref: Optional[str] = None
    requested_courses: list = field(default_factory=list); scenario: str = ""

    @property
    def earned_credits(self):
        return self.other_credits + sum(COURSES[c].credits or 0 for c, g in self.completed.items() if c in COURSES and g != "F")

    def passed(self, code, minimum="D"):
        g = self.completed.get(code); return g is not None and g != "F" and grade_ok(g, minimum)

def load_profiles(path):
    keys = StudentProfile.__dataclass_fields__.keys()
    return [StudentProfile(**{k: v for k, v in d.items() if k in keys}) for d in json.loads(Path(path).read_text(encoding="utf-8"))]

def year_from_credits(earned):
    p = RULES["R7"].params
    return 4 if earned >= p["year4_from"] else 3 if earned >= p["year3_from"] else 2 if earned >= p["year2_from"] else 1

## 5. Rule engine (explicit Python checks)
Every check returns `Finding(rule, level, message)` where `level` is
* **BLOCK** – the course violates a rule and is never recommended,
* **FLAG** – needs a human decision (advisor approval) or the catalog lacks a required detail → not recommended automatically,
* **INFO** – informational (e.g. a retake).

Credit-load, timetable, corequisite and heavy-workload rules (R1–R3, R9, R10, R13) apply to the *whole schedule* and are enforced in the schedule builder and re-verified by `validate_plan()`.

In [5]:
@dataclass
class Finding:
    rule: str; level: str; msg: str

def check_course(c: Course, s: StudentProfile):
    """Course-level rules: R4, R5, R6, R8, R11, R12."""
    F = []
    # R12 - incomplete catalog record: never guess
    needed = [m for m in ("offered", "prerequisites", "schedule") if m in c.missing]
    if needed:
        F.append(Finding("R12", "FLAG", f"Catalog does not publish: {', '.join(needed)}. Eligibility cannot be verified; refer to advisor."))
        return F
    # R8 - availability
    if s.semester not in c.offered:
        F.append(Finding("R8", "BLOCK", f"Not offered in {s.semester} (offered: {', '.join(c.offered)})."))
    # R5 - repeat
    if c.code in s.completed:
        g = s.completed[c.code]
        if grade_ok(g, RULES["R5"].params["repeat_below"]) and g != "F":
            F.append(Finding("R5", "BLOCK", f"Already passed with grade {g}; cannot be repeated."))
        else:
            F.append(Finding("R5", "INFO", f"Retake allowed: previous grade {g} is below {RULES['R5'].params['repeat_below']}."))
    # R4 - prerequisites with minimum grade
    minimum = RULES["R4"].params["min_prereq_grade"]
    for p in c.prereqs:
        g = s.completed.get(p)
        if g is None:
            F.append(Finding("R4", "BLOCK", f"Prerequisite {p} ({COURSES[p].title}) has not been completed."))
        elif not grade_ok(g, minimum):
            F.append(Finding("R4", "BLOCK", f"Prerequisite {p} completed with grade {g}, below the required minimum {minimum}."))
    # R6 - level restriction
    step = int(RULES["R6"].params["level_step"]); allowed = s.academic_year * step
    if c.level > allowed + step:
        F.append(Finding("R6", "BLOCK", f"Level {c.level} is two or more levels above Year {s.academic_year} (max allowed {allowed})."))
    elif c.level > allowed:
        F.append(Finding("R6", "FLAG", f"Level {c.level} is one level above Year {s.academic_year}; requires advisor approval (R14)."))
    # R11 - minimum earned credits
    if c.min_earned and s.earned_credits < c.min_earned:
        F.append(Finding("R11", "BLOCK", f"Requires {c.min_earned} earned credit hours; student has {s.earned_credits}."))
    return F

def status_of(findings):
    return "BLOCKED" if any(f.level == "BLOCK" for f in findings) else "FLAGGED" if any(f.level == "FLAG" for f in findings) else "ELIGIBLE"

def credit_policy(s: StudentProfile):
    """Whole-schedule credit rules R1-R3 (+ R7 year check, R13 note when GPA missing)."""
    r1, r2, r3 = RULES["R1"].params, RULES["R2"].params, RULES["R3"].params
    notes, limit, overload = [], int(r1["max_credits"]), False
    if s.gpa is None:
        notes.append(Finding("R3", "FLAG", "GPA not provided: probation (R3) and overload (R2) eligibility cannot be checked; standard limit R1 applied."))
        notes.append(Finding("R13", "FLAG", "GPA not provided: heavy-course limit (R13) cannot be checked fully; conservative limit of 1 Heavy course applied."))
    elif s.gpa < r3["gpa_below"]:
        limit = int(r3["max_credits"])
        notes.append(Finding("R3", "BLOCK", f"GPA {s.gpa} is below {r3['gpa_below']} (probation): maximum load is {limit} credit hours."))
    elif s.desired_credits > limit and s.gpa >= r2["min_gpa"]:
        limit = min(s.desired_credits, int(r2["max_credits_overload"])); overload = limit > r1["max_credits"]
        notes.append(Finding("R2", "FLAG", f"GPA {s.gpa} qualifies for an overload up to {int(r2['max_credits_overload'])} credits; advisor approval required if more than {int(r1['max_credits'])}."))
    elif s.desired_credits > limit:
        notes.append(Finding("R2", "BLOCK", f"Overload requires GPA >= {r2['min_gpa']}; GPA is {s.gpa}."))
    if s.desired_credits > limit:
        notes.append(Finding("R1" if s.gpa is None or s.gpa >= r3["gpa_below"] else "R3", "BLOCK",
                             f"Requested {s.desired_credits} credits exceeds the allowed maximum of {limit}; load capped at {limit}."))
    if s.desired_credits < r1["min_credits"]:
        notes.append(Finding("R1", "FLAG", f"Requested {s.desired_credits} credits is below the full-time minimum {int(r1['min_credits'])}; part-time status."))
    if year_from_credits(s.earned_credits) != s.academic_year:
        notes.append(Finding("R7", "FLAG", f"Stated Year {s.academic_year} does not match {s.earned_credits} earned credits (Year {year_from_credits(s.earned_credits)}); flagged for registrar."))
    return {"limit": limit, "target": min(s.desired_credits, limit), "overload": overload, "notes": notes}

def heavy_limit(s):
    """Max Heavy courses allowed. Missing GPA → conservative low-GPA limit (not checkable fully)."""
    p = RULES["R13"].params
    if s.gpa is None:
        return int(p["max_heavy_low_gpa"])
    return int(p["max_heavy_low_gpa"]) if s.gpa < p["gpa_threshold"] else int(p["max_heavy"])

def conflicts(a: Course, b: Course):
    return bool(a.schedule and b.schedule and a.schedule.overlaps(b.schedule))

def validate_plan(s: StudentProfile, plan_codes, policy):
    """Independent re-check of the FINAL schedule. Returns a list of violations ([] = pass)."""
    errs, cs = [], [COURSES[c] for c in plan_codes]
    total = sum(c.credits for c in cs)
    if total > policy["limit"]: errs.append(f"R1-R3: total {total} exceeds limit {policy['limit']}")
    for c in cs:
        for f in check_course(c, s):
            if f.level in ("BLOCK", "FLAG"): errs.append(f"{f.rule}: {c.code} {f.msg}")
        for q in c.coreqs:
            if not (s.passed(q) or q in plan_codes): errs.append(f"R10: {c.code} needs corequisite {q}")
    for i, a in enumerate(cs):
        for b in cs[i+1:]:
            if conflicts(a, b): errs.append(f"R9: {a.code} ({a.schedule}) overlaps {b.code} ({b.schedule})")
    if sum(c.workload == "Heavy" for c in cs) > heavy_limit(s): errs.append(f"R13: more than {heavy_limit(s)} Heavy courses")
    return errs

print("Rule engine ready. Parameters loaded from the policy file:",
      {k: {kk: vv for kk, vv in r.params.items()} for k, r in RULES.items() if r.params and r.params != {'none': 'none'} and k in ('R1','R2','R3','R4','R13')})


Rule engine ready. Parameters loaded from the policy file: {'R1': {'min_credits': 12.0, 'max_credits': 18.0}, 'R2': {'min_gpa': 3.5, 'max_credits_overload': 21.0}, 'R3': {'gpa_below': 2.0, 'max_credits': 12.0}, 'R4': {'min_prereq_grade': 'C'}, 'R13': {'max_heavy': 2.0, 'gpa_threshold': 2.5, 'max_heavy_low_gpa': 1.0}}


## 6. LLM layer (any pretrained model or API)
`call_llm()` prefers the **free NVIDIA NIM** API (`NVIDIA_API_KEY` → `https://integrate.api.nvidia.com/v1/chat/completions`, default model `nvidia/nemotron-3.5-bolt-e4b`). It also accepts **OpenAI** (`OPENAI_API_KEY`) or **Anthropic** (`ANTHROPIC_API_KEY`). Calls use plain HTTPS – no SDK needed. If no key is configured, it returns `None` and the system uses a deterministic **evidence-grounded template** instead.
A **guardrail** rejects any LLM output that mentions course codes outside the catalog or omits a recommended course, then falls back to the template.


In [6]:
def _openai_compatible_chat(url, api_key, model, system, user, max_tokens):
    body = {"model": model, "max_tokens": max_tokens,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    req = urllib.request.Request(url, json.dumps(body).encode(),
          {"content-type": "application/json", "authorization": "Bearer " + api_key})
    return json.load(urllib.request.urlopen(req, timeout=90))["choices"][0]["message"]["content"]

def call_llm(system, user, max_tokens=900):
    """Try NVIDIA NIM (free) → OpenAI → Anthropic. Return None on failure / no key."""
    try:
        if os.getenv("NVIDIA_API_KEY"):
            return _openai_compatible_chat(
                "https://integrate.api.nvidia.com/v1/chat/completions",
                os.environ["NVIDIA_API_KEY"],
                os.getenv("NVIDIA_MODEL", "nvidia/nemotron-3.5-bolt-e4b"),
                system, user, max_tokens)
        if os.getenv("OPENAI_API_KEY"):
            return _openai_compatible_chat(
                "https://api.openai.com/v1/chat/completions",
                os.environ["OPENAI_API_KEY"],
                os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
                system, user, max_tokens)
        if os.getenv("ANTHROPIC_API_KEY"):
            body = {"model": os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5"), "max_tokens": max_tokens, "system": system,
                    "messages": [{"role": "user", "content": user}]}
            req = urllib.request.Request("https://api.anthropic.com/v1/messages", json.dumps(body).encode(),
                  {"content-type": "application/json", "x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
            return json.load(urllib.request.urlopen(req, timeout=90))["content"][0]["text"]
    except Exception as e:
        print("  [LLM call failed -> using offline template]", type(e).__name__, str(e)[:120])
    return None

def llm_provider_label():
    if os.getenv("NVIDIA_API_KEY"):
        return f"NVIDIA NIM ({os.getenv('NVIDIA_MODEL', 'nvidia/nemotron-3.5-bolt-e4b')})"
    if os.getenv("OPENAI_API_KEY"):
        return f"OpenAI ({os.getenv('OPENAI_MODEL', 'gpt-4o-mini')})"
    if os.getenv("ANTHROPIC_API_KEY"):
        return f"Anthropic ({os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-5')})"
    return None

LLM_PROVIDER = llm_provider_label()
LLM_AVAILABLE = LLM_PROVIDER is not None
print("LLM available:", LLM_AVAILABLE, f"({LLM_PROVIDER})" if LLM_PROVIDER else "(offline template used otherwise)")


LLM available: True (NVIDIA NIM (nvidia/nemotron-3.5-bolt-e4b))


## 7. Recommendation engine
1. **Retrieve** – build a query from the student's interests, goals, skills; retrieve the course passages (ranked by cosine similarity) and degree-plan passages.
2. **Check rules** – run `check_course()` on every catalog course → ELIGIBLE / FLAGGED / BLOCKED. For each finding, the matching policy passage is *retrieved* from the index (rank and score are reported) as evidence.
3. **Rank** eligible courses: `score = 0.60·relevance + 0.25·program_priority + 0.10·workload_fit + 0.05·requested`.
4. **Build the schedule** greedily under R1–R3 (credits), R9 (timetable), R10 (corequisites), R13 (heavy limit).
5. **Validate** the final schedule again with independent checks, then **generate** the explanations.

In [7]:
def build_query(s):
    return " ".join(filter(None, [", ".join(s.interests), s.goals, ", ".join(s.skills)]))

def interest_matches(s, c: Course):
    """Match interests/skills to a course via substring or token overlap (not brittle 5-char stems)."""
    hay = (c.title + " " + c.description + " " + " ".join(c.topics)).lower()
    hay_toks = set(re.findall(r"[a-z]{3,}", hay))
    out = []
    for phrase in list(s.interests) + list(s.skills):
        pl = phrase.lower().strip()
        if len(pl) < 3: continue
        if pl in hay:
            out.append(phrase); continue
        toks = re.findall(r"[a-z]{3,}", pl)
        if not toks: continue
        ok = True
        for t in toks:
            if t in hay_toks or any(h.startswith(t[:4]) or t.startswith(h[:4]) for h in hay_toks):
                continue
            ok = False; break
        if ok: out.append(phrase)
    return out

def program_priority(c: Course, s: StudentProfile):
    if c.category.startswith("Core"): return 1.0 if (c.core_year or 9) <= s.academic_year else 0.3
    if c.category.startswith("General Education"): return 0.6
    return 0.0

def workload_fit(c, pref):
    lv = {"light": 0, "medium": 1, "heavy": 2}
    if pref is None: return 0.5
    return 1 - abs(lv[pref] - lv[c.workload.lower()]) / 2

def rule_evidence(finding: Finding):
    """RAG step for rules: retrieve the policy passage that explains this finding and report where the expected rule ranked."""
    hits = INDEX.search(f"{RULES[finding.rule].title}. {finding.msg}", k=3, kind="rule")
    rank = next((i + 1 for i, (ch, _) in enumerate(hits) if ch.cid == finding.rule), None)
    score = next((sc for ch, sc in hits if ch.cid == finding.rule), None)
    return {"rule": finding.rule, "source": RULES[finding.rule].source, "rule_title": RULES[finding.rule].title,
            "policy_text": RULES[finding.rule].text, "retrieved_rank": rank, "retrieval_score": round(score, 3) if score is not None else None}

def recommend(s: StudentProfile):
    query = build_query(s)
    hits = INDEX.search(query, k=len(CHUNKS), kind="course")
    rel = {}
    for ch, sc in hits: rel[ch.cid.split("-")[0]] = max(rel.get(ch.cid.split("-")[0], 0), sc)
    rank_of = {code: i + 1 for i, code in enumerate(sorted(rel, key=rel.get, reverse=True))}
    plan_hits = INDEX.search(query, k=2, kind="plan")
    policy = credit_policy(s)

    audit = {c: check_course(COURSES[c], s) for c in COURSES}
    elig = [c for c in COURSES if status_of(audit[c]) == "ELIGIBLE" and not (s.passed(c, RULES["R5"].params["repeat_below"]))]
    maxrel = max([rel[c] for c in elig] + [1e-9])
    score = {c: round(0.60 * rel[c] / maxrel + 0.25 * program_priority(COURSES[c], s) + 0.10 * workload_fit(COURSES[c], s.workload_pref)
                      + 0.05 * (c in s.requested_courses), 4) for c in elig}
    ranked = sorted(elig, key=lambda c: -score[c])

    selected, credits, heavy, skip = [], 0, 0, {}
    def reasons_to_skip(c, extra_codes=()):
        co = COURSES[c]; R = []
        group = [co] + [COURSES[q] for q in co.coreqs if not s.passed(q) and q not in selected]
        need = sum(g.credits for g in group)
        for q in co.coreqs:
            if not s.passed(q) and q not in selected and q not in elig:
                R.append(Finding("R10", "BLOCK", f"Corequisite {q} is not eligible this semester, so {c} cannot be registered."))
        if credits + need > policy["target"]:
            R.append(Finding("R1", "BLOCK", f"Adding it ({need} cr) would raise the total to {credits+need}, above the {policy['target']}-credit limit/target."))
        for g in group:
            for x in selected + [e for e in extra_codes]:
                if conflicts(g, COURSES[x]):
                    R.append(Finding("R9", "BLOCK", f"Timetable conflict: {g.code} ({g.schedule}) overlaps {x} ({COURSES[x].schedule})."))
        if sum(g.workload == "Heavy" for g in group) + heavy > heavy_limit(s):
            R.append(Finding("R13", "BLOCK", f"Would exceed the limit of {heavy_limit(s)} Heavy course(s) for this student."))
        return R, group
    for c in ranked:
        if credits >= policy["target"]: skip[c] = [Finding("R1", "BLOCK", f"Credit target of {policy['target']} already reached.")]; continue
        R, group = reasons_to_skip(c)
        if R: skip[c] = R; continue
        for g in group:
            selected.append(g.code); credits += g.credits; heavy += g.workload == "Heavy"
            if g.code != c: skip.pop(g.code, None)
    errs = validate_plan(s, selected, policy)
    if credits < RULES["R1"].params["min_credits"]:
        policy["notes"].append(Finding("R1", "FLAG", f"Only {credits} eligible credit hours could be scheduled, below the full-time minimum {int(RULES['R1'].params['min_credits'])}; advisor should review (part-time status)."))

    # ----- which non-selected courses to report -----
    top_rel = max(rel.values())
    relevant = [c for c in sorted(rel, key=rel.get, reverse=True)[:TOP_K_COURSES] if rel[c] >= 0.25 * top_rel]   # ignore near-zero similarity noise
    constraint_hits = [c for c in skip if any(x.rule in ("R9", "R10", "R13") for x in skip[c])]               # eligible courses lost to schedule rules
    show = list(dict.fromkeys(s.requested_courses + relevant + constraint_hits))
    excluded = []
    for c in show:
        if c in selected: continue
        if s.passed(c, RULES["R5"].params["repeat_below"]) and c not in s.requested_courses: continue
        f = [x for x in audit[c] if x.level in ("BLOCK", "FLAG")]
        if not f and c in skip: f = skip[c]
        if not f: f = [Finding("R1", "INFO", "Eligible but lower priority than the selected courses.")]
        st = status_of(audit[c]) if status_of(audit[c]) != "ELIGIBLE" else "ELIGIBLE (not selected)"
        excluded.append({"code": c, "title": COURSES[c].title, "status": st, "retrieval_rank": rank_of.get(c),
                         "reasons": [{"rule": x.rule, "level": x.level, "message": x.msg, "evidence": rule_evidence(x)} for x in f]})
    # ----- accepted courses -----
    accepted = []
    for c in selected:
        co = COURSES[c]
        rules_used = [Finding("R4", "OK", ("Prerequisites met: " + ", ".join(f"{p} ({s.completed[p]})" for p in co.prereqs)) if co.prereqs else "No prerequisites."),
                      Finding("R6", "OK", f"Level {co.level} allowed for Year {s.academic_year}."),
                      Finding("R8", "OK", f"Offered in {s.semester}."),
                      Finding("R9", "OK", f"Meets {co.schedule}; no overlap with the other selected courses.")]
        if co.coreqs: rules_used.append(Finding("R10", "OK", f"Corequisite satisfied: {', '.join(co.coreqs)}."))
        if co.min_earned: rules_used.append(Finding("R11", "OK", f"{s.earned_credits} earned credits >= required {co.min_earned}."))
        if c in s.completed: rules_used.append(Finding("R5", "INFO", f"Retake (previous grade {s.completed[c]})."))
        ch = next(ch for ch in CHUNKS if ch.cid == c)
        accepted.append({"code": c, "title": co.title, "credits": co.credits, "level": co.level, "workload": co.workload, "schedule": str(co.schedule),
                         "category": co.category, "score": score[c], "relevance_rank": rank_of[c], "matched_interests": interest_matches(s, co),
                         "evidence": {"course_source": co.source, "retrieval_score": round(rel[c], 3),
                                      "rules_checked": [{"rule": f.rule, "result": f.level, "detail": f.msg, "source": RULES[f.rule].source} for f in rules_used]}})
    notes = []
    if credits < policy["target"]:
        notes.append(f"Scheduled {credits} of the {policy['target']} target credit hours: no remaining eligible course fits the credit limit, timetable or heavy-workload rules.")
    if policy["limit"] > RULES["R1"].params["max_credits"] and credits <= RULES["R1"].params["max_credits"]:
        notes.append(f"Overload (R2) was available but is not needed: the schedule has {credits} credit hours, within the standard limit.")
    result = {"student": {k: v for k, v in asdict(s).items()} | {"earned_credits": s.earned_credits},
              "credit_policy": {"limit": policy["limit"], "target": policy["target"], "overload_needs_approval": policy["overload"],
                                "notes": [{"rule": f.rule, "level": f.level, "message": f.msg, "evidence": rule_evidence(f)} for f in policy["notes"]]},
              "retrieval": {"backend": INDEX.name, "query": query,
                            "top_courses": [{"rank": i + 1, "code": c, "score": round(rel[c], 3), "source": COURSES[c].source}
                                            for i, c in enumerate(sorted(rel, key=rel.get, reverse=True)[:6])],
                            "degree_plan_passages": [{"source": ch.source, "score": round(sc, 3), "text": ch.text} for ch, sc in plan_hits]},
              "accepted": accepted, "excluded": excluded, "total_credits": credits,
              "uncovered_interests": [i for i in s.interests if not any(i in a["matched_interests"] for a in accepted)],
              "notes": notes, "validation": {"passed": not errs, "violations": errs},
              "full_audit": {c: {"status": status_of(audit[c]), "findings": [f"{x.rule} {x.level}: {x.msg}" for x in audit[c]]} for c in COURSES}}
    explain(s, result)
    return result

# ---------- explanations (LLM with guardrail, template fallback) ----------
def template_reason(s, a):
    co, parts = COURSES[a["code"]], []
    if a["matched_interests"]: parts.append(f"Matches your interests/skills: {', '.join(a['matched_interests'])} (topics: {'; '.join(co.topics[:3])}).")
    if co.category.startswith("Core"): parts.append(f"Core course planned for Year {co.core_year} of the degree plan" + ("." if (co.core_year or 9) <= s.academic_year else " (you are ahead of schedule)."))
    elif co.category.startswith("General"): parts.append("Required general-education course.")
    else: parts.append(f"{co.category.replace(' Elective','')} elective that broadens your profile.")
    if a["code"] in s.completed: parts.append(f"Retake to improve your earlier grade ({s.completed[a['code']]}).")
    parts.append(("Prerequisites satisfied (" + ", ".join(f"{p}: {s.completed[p]}" for p in co.prereqs) + ").") if co.prereqs else "No prerequisites.")
    parts.append(f"{co.workload} workload" + (f", fits your {s.workload_pref} preference." if co.workload.lower() == s.workload_pref else "."))
    return " ".join(parts)

def _parse_llm_reasons(txt):
    """Parse CODE: reason lines; tolerate bold, dashes, and extra spaces."""
    got = {}
    for m in re.finditer(r"(?im)^\s*\**([A-Z]{2,5}\d{3})\**\s*[:\-–]\s*(.+?)\s*$", txt):
        got[m.group(1).upper()] = m.group(2).strip().strip("*").strip()
    return got

def explain(s, result):
    for a in result["accepted"]: a["reason"] = template_reason(s, a)
    result["explanation_mode"] = "template (offline)"
    if not (LLM_AVAILABLE and result["accepted"]): return
    ctx = {"student": {k: result["student"][k] for k in ("name", "academic_year", "semester", "gpa", "interests", "skills", "goals", "workload_pref")},
           "accepted_courses": [{"code": a["code"], "title": a["title"], "category": a["category"], "workload": a["workload"], "matched_interests": a["matched_interests"],
                                 "catalog_passage": next(ch.text for ch in CHUNKS if ch.cid == a["code"]),
                                 "rules": [r["detail"] for r in a["evidence"]["rules_checked"]]} for a in result["accepted"]]}
    codes = ", ".join(a["code"] for a in result["accepted"])
    system = ("You are a university course advisor. Use ONLY the facts in the JSON context. Write one concise reason (max 35 words) per accepted course, "
              f"linking it to the student's knowledge, interests or goals. Output EXACTLY these course codes, one per line, as `CODE: reason`: {codes}. "
              "Do not mention any other course code.")
    for attempt in range(2):
        txt = call_llm(system, json.dumps(ctx, indent=1))
        if not txt: return
        got = _parse_llm_reasons(txt)
        ok = all(a["code"] in got for a in result["accepted"]) and all(set(re.findall(r"\b[A-Z]{2,5}\d{3}\b", r)) <= set(COURSES) for r in got.values())
        if ok:
            for a in result["accepted"]: a["reason"] = got[a["code"]].strip()
            result["explanation_mode"] = f"LLM via {LLM_PROVIDER} (grounded + guardrail passed)"
            return
        if attempt == 0:
            print("  [guardrail] retrying LLM explanation once")
    print("  [guardrail] LLM output rejected -> template used")
print("Recommendation engine ready.")


Recommendation engine ready.


## 8. Report formatter and the 6 test profiles
For every profile the notebook prints: **input → retrieved evidence → rule findings → accepted courses (with sources) → pending/excluded courses with reasons → total credits → rule audit**.

In [8]:
def format_result(r, width=118):
    st, L = r["student"], []
    w = lambda t, ind="    ": L.append(textwrap.fill(t, width, initial_indent=ind, subsequent_indent=ind + "  "))
    L.append("=" * width)
    L.append(f"STUDENT {st['student_id']} – {st['name']}   |   Year {st['academic_year']}, {st['semester']} semester   |   GPA {st['gpa'] if st['gpa'] is not None else 'n/a'}   |   earned credits {st['earned_credits']}")
    L.append("=" * width)
    if st.get("scenario"): w("Scenario: " + st["scenario"], "  ")
    L.append("\n1) INPUT")
    comp = ", ".join(f"{c}({g})" for c, g in st["completed"].items()) or "none"
    for line in (f"Completed: {comp}", f"Skills: {', '.join(st['skills'])}", f"Interests: {', '.join(st['interests'])}  |  Goal: {st['goals']}",
                 f"Desired credits: {st['desired_credits']}  |  Workload preference: {st['workload_pref']}  |  Explicitly requested: {', '.join(st['requested_courses']) or '-'}"):
        w(line)
    rt = r["retrieval"]
    L.append(f"\n2) RETRIEVED EVIDENCE   (backend: {rt['backend']})")
    w(f"Query: \"{rt['query']}\"")
    w("Top retrieved course passages: " + "; ".join(f"#{t['rank']} {t['code']} ({t['score']})" for t in rt["top_courses"]))
    for p in rt["degree_plan_passages"]: w(f"Degree-plan passage [{p['source']}] ({p['score']}): " + textwrap.shorten(p["text"], 230))
    cp = r["credit_policy"]
    L.append(f"\n3) CREDIT-LOAD RULES   (limit {cp['limit']}, target {cp['target']})")
    if not cp["notes"]: w("No credit-load restriction triggered (R1 standard load 12–18 applies).")
    for n in cp["notes"]:
        e = n["evidence"]; w(f"[{n['rule']}/{n['level']}] {n['message']}")
        w(f"Evidence {e['source']} (retrieved rank {e['retrieved_rank'] or 'n/a – direct lookup'}): {textwrap.shorten(e['policy_text'], 170)}", "        ")
    L.append(f"\n4) ACCEPTED COURSES  → total {r['total_credits']} credit hours")
    for a in r["accepted"]:
        L.append(f"  ✔ {a['code']}  {a['title']}  ({a['credits']} cr, {a['workload']}, {a['schedule']})  score {a['score']}")
        w(a["reason"], "      ")
        w("Sources: " + a["evidence"]["course_source"] + f" (sim {a['evidence']['retrieval_score']}); rules " + ", ".join(sorted({x['rule'] for x in a['evidence']['rules_checked']}, key=lambda z: int(z[1:]))), "      ")
    if not r["accepted"]: w("No eligible course could be scheduled.")
    flagged = [e for e in r["excluded"] if e["status"] == "FLAGGED"]; rest = [e for e in r["excluded"] if e["status"] != "FLAGGED"]
    if flagged:
        L.append("\n5) FLAGGED – NEEDS ADVISOR (not auto-recommended)")
        for e in flagged:
            L.append(f"  ⚑ {e['code']}  {e['title']}")
            for x in e["reasons"]: w(f"[{x['rule']}] {x['message']}", "      ")
    L.append("\n6) EXCLUDED COURSES (among requested / top-retrieved)")
    for e in rest:
        L.append(f"  ✘ {e['code']}  {e['title']}  – {e['status']}")
        for x in e["reasons"]:
            ev = x["evidence"]; w(f"[{x['rule']}] {x['message']}  (evidence {ev['source']}, retrieved rank {ev['retrieved_rank'] or 'n/a'})", "      ")
    for n in r.get("notes", []): w("Note: " + n, "  ")
    if r["uncovered_interests"]: L.append("\n  Interests not covered by any eligible course this semester: " + ", ".join(r["uncovered_interests"]))
    L.append(f"\n7) RULE AUDIT (independent re-check of final schedule): {'PASSED' if r['validation']['passed'] else 'FAILED – ' + str(r['validation']['violations'])}"
             f"   |   explanations: {r['explanation_mode']}")
    return "\n".join(L)

PROFILES = load_profiles(DATA_DIR / "student_profiles.json")
RESULTS = {}
for p in PROFILES:
    RESULTS[p.student_id] = recommend(p)
    print(format_result(RESULTS[p.student_id])); print()

STUDENT S001 – Sara Mostafa   |   Year 1, Fall semester   |   GPA n/a   |   earned credits 0
  Scenario: New first-year student, no GPA yet; asks for a course she is not eligible for (level + prerequisites).

1) INPUT
    Completed: none
    Skills: basic Python from online tutorials
    Interests: programming, artificial intelligence, problem solving  |  Goal: become an AI engineer
    Desired credits: 15  |  Workload preference: medium  |  Explicitly requested: AI301

2) RETRIEVED EVIDENCE   (backend: TF-IDF unigram+bigram (scikit-learn fallback))
    Query: "programming, artificial intelligence, problem solving become an AI engineer basic Python from online
      tutorials"
    Top retrieved course passages: #1 AI301 (0.165); #2 CS201 (0.149); #3 CS102 (0.143); #4 CS301 (0.14); #5 AI401
      (0.129); #6 CS101 (0.094)
    Degree-plan passage [degree_plan.md#Artificial Intelligence track] (0.157): Artificial Intelligence track.
      Students interested in machine learning, data scie

### Summary of the six test profiles

In [9]:
print(f"{'ID':5} {'Student':14} {'Yr/Sem':9} {'GPA':4} {'Req':>3} {'Cap':>3} {'Total':>5}  Accepted → ")
for sid, r in RESULTS.items():
    st = r["student"]
    print(f"{sid:5} {st['name'][:14]:14} {st['academic_year']}/{st['semester']:6} {str(st['gpa']):4} {st['desired_credits']:>3} {r['credit_policy']['limit']:>3} {r['total_credits']:>5}  " + ", ".join(a["code"] for a in r["accepted"])
          + f"   [audit {'OK' if r['validation']['passed'] else 'FAIL'}]")

ID    Student        Yr/Sem    GPA  Req Cap Total  Accepted → 
S001  Sara Mostafa   1/Fall   None  15  18    15  CS102, CS101, MATH101, PHYS101, ENG101   [audit OK]
S002  Omar Hassan    2/Spring 3.1   15  18    14  MATH201, STAT201, CS210, CS203, HUM201   [audit OK]
S003  Lina Farouk    3/Fall   3.2   21  18    18  CS370, CS301, CS320, CS310, CS340, AI320   [audit OK]
S004  Karim Adel     4/Fall   3.6   21  21    15  CS401, CS410, SEC301, CS340, AI320   [audit OK]
S005  Nour El-Din    2/Fall   1.8   18  12    12  CS220, CS202, MATH101, CS201   [audit OK]
S006  Maya Samir     3/Spring 3.4   15  18    15  AI310, CS330, CS310, AI320, CS350   [audit OK]


## 9. Chatbot for course registration
A conversational front-end on top of the same RAG + rule engine.
* **Slot filling:** collects year, semester, GPA, completed courses (with grades), interests, goals, desired credits – via an LLM JSON extractor when a key is set, otherwise regex rules.
* **Intents:** recommend a schedule · *"why can't I take X?"* (rule engine + retrieved policy evidence) · *"tell me about X"* · policy questions (retrieval-augmented answer) · change a preference (re-plans) · reset.
* Missing details are **asked for, not invented** (e.g. no GPA → R2/R3/R13 flagged as not checkable; a course listed without a grade is assumed to be "C" and flagged).


In [10]:
ORD = {"first": 1, "freshman": 1, "second": 2, "sophomore": 2, "third": 3, "junior": 3, "fourth": 4, "final": 4, "senior": 4}
CODE_RE = re.compile(r"\b([A-Z]{2,5})\s?(\d{3})\b(?:\s*\(\s*([A-F][+-]?)\s*\)|\s+([A-F][+-]?)(?=[\s,.;)]|$))?")

def regex_slots(text):
    low, out = text.lower(), {}
    m = re.search(r"\b(?:year|level)\s*([1-4])\b", low) or re.search(r"\b([1-4])(?:st|nd|rd|th)[- ]year", low)
    if m: out["academic_year"] = int(m.group(1))
    else:
        m = re.search(r"\b(first|second|third|fourth|final|freshman|sophomore|junior|senior)\b[- ]*(?:year)?", low)
        if m: out["academic_year"] = ORD[m.group(1)]
    m = re.search(r"\b(fall|spring)\b", low)
    if m: out["semester"] = m.group(1).capitalize()
    m = re.search(r"gpa\D{0,12}([0-4](?:\.\d+)?)", low)
    if m: out["gpa"] = float(m.group(1))
    m = re.search(r"(\d{1,3})\s*(?:transfer|other)\s*credit", low)
    if m: out["other_credits"] = int(m.group(1)); low = low.replace(m.group(0), " ")
    m = re.search(r"(\d{1,2})\s*(?:credit|credits|credit hours|cr)\b", low)
    if m: out["desired_credits"] = int(m.group(1))
    if re.search(r"\b(light|easy|relaxed)\b", low): out["workload_pref"] = "light"
    elif re.search(r"\b(heavy|challenging|intense|demanding)\b", low): out["workload_pref"] = "heavy"
    elif re.search(r"\b(medium|moderate|balanced)\b", low): out["workload_pref"] = "medium"
    completed, requested = {}, []
    for clause in re.split(r"[.;!?\n]", text):
        codes = CODE_RE.findall(clause)
        if not codes: continue
        done = re.search(r"\b(completed|passed|finished|took|taken|done|already|have)\b", clause, re.I) and not re.search(r"\b(want|can i|why|register|eligible|allowed)\b", clause, re.I)
        for pre, num, g1, g2 in codes:
            code = pre + num
            if code not in COURSES: continue
            if done: completed[code] = g1 or g2 or None
            else: requested.append(code)
    if completed: out["completed"] = completed
    if requested: out["requested_courses"] = requested
    if re.search(r"\b(new student|first semester|just started|incoming|no (?:completed )?courses|haven'?t (?:taken|completed)|nothing yet)\b", low): out.setdefault("completed", {})
    m = re.search(r"(?:interested in|interests? (?:are|include|is)|i (?:like|love|enjoy)|passionate about|want to (?:learn|study|focus on))\s+([^.;!?\n]+)", text, re.I)
    if m:
        seg = re.split(r"\b(?:and i(?:'d| would| want)|i(?:'d| would) like|i want \d|with a |but |my goal)", m.group(1), flags=re.I)[0]
        items = [x.strip(" ,") for x in re.split(r",| and ", seg) if len(x.strip(" ,")) > 2]
        if items: out["interests"] = items
    m = re.search(r"(?:my goal(?:s)? (?:is|are)|aiming for|i want to (?:become|work|pursue)|career in|goal[:\s]+)\s*([^.;!?\n]+)", text, re.I)
    if m:
        g = m.group(1).strip(" .,")
        if len(g) > 3: out["goals"] = g
    return out

def llm_slots(text):
    sys_p = ("Extract registration details from the student's message as JSON only, with keys (omit unknown ones): academic_year (1-4), semester (Fall|Spring), gpa (float), "
             "other_credits (int), completed (object code->grade, grade null if unknown), interests (list of short phrases), goals (short string), "
             "desired_credits (int), workload_pref (light|medium|heavy), requested_courses (list of codes). "
             "Only use what the message states.")
    raw = call_llm(sys_p, text, 400)
    try: return json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    except Exception: return None

QUESTIONS = {
    "academic_year": "Which academic year are you in (1–4)?",
    "semester": "Which semester are you registering for — Fall or Spring?",
    "gpa": "What is your current cumulative GPA? (type 'skip' if you don't have one yet)",
    "completed": "Which courses have you completed, with grades? e.g. “CS101 (A), CS102 (B+)” — or say “none” if you are a new student.",
    "interests": "What are you interested in? e.g. “machine learning, web development”.",
    "goals": "What is your academic or career goal? e.g. “work in AI research” (or type 'skip').",
    "desired_credits": "How many credit hours would you like to take?",
}
POLICY_WORDS = r"credit|overload|prerequisite|probation|repeat|retake|rule|policy|limit|maximum|minimum|conflict|level|heavy|approval|corequisite|gpa|offered"

class AdvisorChatbot:
    def __init__(self):
        self.slots, self.last, self.gpa_skipped, self.goals_skipped, self.assumed, self.awaiting = {}, None, False, False, [], None

    # ---- helpers ----
    def _merge(self, new):
        changed = {k: v for k, v in new.items() if v is not None and self.slots.get(k) != v and k != "completed" and k != "requested_courses"}
        if "completed" in new and new["completed"] is not None:
            comp = self.slots.setdefault("completed", {})
            for c, g in new["completed"].items():
                if g is None: g = "C"; self.assumed.append(c)
                if comp.get(c) != g: changed["completed"] = True
                comp[c] = g
        if new.get("requested_courses"): self.slots["requested_courses"] = list(dict.fromkeys(self.slots.get("requested_courses", []) + new["requested_courses"]))
        self.slots.update({k: v for k, v in changed.items() if k != "completed"})
        return changed
    def _missing(self):
        return [k for k in QUESTIONS if (k not in self.slots and not (k == "gpa" and self.gpa_skipped) and not (k == "goals" and self.goals_skipped))]
    def _profile(self):
        s = self.slots
        return StudentProfile("CHAT", s.get("name", "Student"), s["academic_year"], s["semester"], s.get("gpa"), s.get("completed", {}), s.get("desired_credits", 15),
                              s.get("other_credits", 0), s.get("skills", []), s.get("interests", []), s.get("goals", ""), s.get("workload_pref"), s.get("requested_courses", []))
    def _ready(self): return not self._missing()

    # ---- answers ----
    def _plan_text(self, r):
        st = r["student"]
        L = [f"Here is a feasible {st['semester']} plan for Year {st['academic_year']} — {r['total_credits']} credit hours (limit {r['credit_policy']['limit']}):"]
        for a in r["accepted"]:
            L.append(f"  • {a['code']} {a['title']} ({a['credits']} cr, {a['schedule']}) — {a['reason']}")
        pend = [e for e in r["excluded"] if e["status"] == "FLAGGED"]
        out = [e for e in r["excluded"] if e["status"] != "FLAGGED" and e["code"] in st["requested_courses"] + [t["code"] for t in r["retrieval"]["top_courses"][:4]]]
        if pend: L.append("Needs advisor approval (not included): " + "; ".join(f"{e['code']} ({e['reasons'][0]['rule']}: {e['reasons'][0]['message'][:80]})" for e in pend))
        if out:
            L.append("Not recommended:")
            for e in out[:4]: L.append(f"  ✘ {e['code']} — " + " ".join(f"[{x['rule']}] {x['message']}" for x in e["reasons"][:2]))
        for n in r["credit_policy"]["notes"]: L.append(f"⚠ [{n['rule']}] {n['message']}")
        for n in r["notes"]: L.append("ℹ " + n)
        if self.assumed: L.append(f"⚠ Grades not given for {', '.join(self.assumed)} — assumed 'C' (minimum prerequisite grade); please confirm.")
        L.append("Sources: " + ", ".join(sorted({a["evidence"]["course_source"] for a in r["accepted"]} | {"registration_rules.md"})))
        L.append(f"Rule audit: {'passed' if r['validation']['passed'] else 'FAILED'}. Ask “why can't I take <course>?”, change a preference, or say “reset”.")
        return "\n".join(L)

    def explain_course(self, code):
        p = self._profile(); c = COURSES[code]
        F = check_course(c, p); blocks = [f for f in F if f.level in ("BLOCK", "FLAG")]
        if not blocks:
            return f"{code} ({c.title}) is allowed for you in {p.semester}: prerequisites met, level OK, offered, schedule {c.schedule}. [{c.source}] Whether it fits depends on your credit limit and timetable."
        L = [f"{code} ({c.title}) cannot be registered automatically:"]
        for f in blocks:
            e = rule_evidence(f)
            L.append(f"  • [{f.rule}] {f.msg}\n      Policy ({e['source']}): {textwrap.shorten(e['policy_text'], 150)}")
        return "\n".join(L)

    def course_info(self, code):
        c = COURSES[code]; ch = next(x for x in CHUNKS if x.cid == code)
        return f"{ch.text}  [source: {c.source}]  Schedule: {c.schedule}"

    def policy_answer(self, q):
        if self._ready() and re.search(r"credit|overload|maximum|limit|how many", q.lower()):       # personalised answer from the governing rule
            pr = self._profile(); cp = credit_policy(pr); g = pr.gpa
            rid = "R3" if (g is not None and g < RULES["R3"].params["gpa_below"]) else "R2" if (g is not None and g >= RULES["R2"].params["min_gpa"]) else "R1"
            base = f"For your profile (GPA {g if g is not None else 'unknown'}) the standard maximum is {int(RULES['R1'].params['max_credits'])} credit hours; applying the rules gives a limit of {cp['limit']}" + (" (the part above 18 needs advisor approval)." if rid == "R2" else ".")
            return f"{base}\n  Governing policy [{RULES[rid].source}]: {textwrap.shorten(RULES[rid].text, 260)}"
        # query rewriting: map the question to the policy topic it asks about, so retrieval is robust to wording
        TOPIC = [(r"credit|overload|maximum|limit|probation|how many", "credit hours per semester standard load overload probation"),
                 (r"prerequisite|grade", "prerequisites minimum grade"), (r"repeat|retake", "repeating courses"),
                 (r"level|year|approval", "course level restriction advisor approval"), (r"conflict|timetable|clash", "timetable conflicts overlap"),
                 (r"heavy|workload", "heavy workload limit"), (r"offered|semester", "course availability by semester")]
        q2 = next((t for pat, t in TOPIC if re.search(pat, q.lower())), q)
        hits = INDEX.search(q2, k=2, kind='rule') + INDEX.search(q, k=1, kind='plan')
        ctx = "\n".join(f"[{ch.source}] {ch.text}" for ch, _ in hits)
        extra = ""
        if self._ready() and re.search(r"credit|overload|maximum|limit", q.lower()):
            cp = credit_policy(self._profile()); extra = f"\nFor your profile: maximum {cp['limit']} credit hours" + ("." if not cp["overload"] else " (overload needs advisor approval).")
        ans = call_llm("You answer questions about NVU registration using ONLY the provided passages. Be concise and cite the source tag in brackets. If the passages do not contain the answer, say so.",
                       f"Passages:\n{ctx}\n\nQuestion: {q}", 300) if LLM_AVAILABLE else None
        if not ans:
            top, _ = hits[0]
            ans = f"{textwrap.shorten(top.text, 330)}  [source: {top.source}]"
        return ans + extra

    # ---- main entry ----
    def reply(self, text):
        low = text.lower().strip()
        if re.search(r"\b(reset|start over|new student)\b", low) and len(low) < 40:
            self.__init__(); return "Okay, starting over. " + QUESTIONS["academic_year"]
        new = (llm_slots(text) if LLM_AVAILABLE else None) or regex_slots(text)
        if self.awaiting == "gpa" and "gpa" not in new:
            if re.search(r"skip|don'?t know|not sure|n/?a|no gpa|none|new", low): self.gpa_skipped = True
            elif re.fullmatch(r"\s*[0-4](\.\d+)?\s*", low): new["gpa"] = float(low)
        if self.awaiting == "goals" and "goals" not in new:
            if re.search(r"skip|don'?t know|not sure|n/?a|none|no goal", low): self.goals_skipped = True
            elif len(low) > 3: new["goals"] = text.strip()
        if self.awaiting == "desired_credits" and "desired_credits" not in new and re.fullmatch(r"\s*\d{1,2}\s*", low): new["desired_credits"] = int(low)
        if self.awaiting == "academic_year" and "academic_year" not in new and re.fullmatch(r"\s*[1-4]\s*", low): new["academic_year"] = int(low)
        if self.awaiting == "interests" and "interests" not in new and len(low) < 120: new["interests"] = [x.strip() for x in re.split(r",| and ", low) if x.strip()]
        codes = [m[0] + m[1] for m in CODE_RE.findall(text) if m[0] + m[1] in COURSES]
        changed = self._merge(new)
        # intents
        if codes and re.search(r"\b(why|can i|could i|eligible|allowed|able to)\b", low) and "academic_year" in self.slots and "semester" in self.slots:
            self.slots.setdefault("completed", {}); self.awaiting = None
            return "\n".join(self.explain_course(c) for c in dict.fromkeys(codes))
        if codes and re.search(r"\b(tell me about|what is|describe|about)\b", low):
            return "\n".join(self.course_info(c) for c in dict.fromkeys(codes))
        if "?" in text and re.search(POLICY_WORDS, low) and not (changed and self._ready()):
            return self.policy_answer(text)
        if self._ready() and (self.last is None or changed or re.search(r"\b(recommend|suggest|plan|schedule|register|again|redo|update)\b", low)):
            self.awaiting = None
            self.last = recommend(self._profile())
            return self._plan_text(self.last)
        missing = self._missing()
        if not missing: return "Tell me what you'd like: a plan (“recommend courses”), an eligibility check (“why can't I take AI301?”), or a rule question."
        self.awaiting = missing[0]
        got = ", ".join((f"{len(v)} completed courses" if k == "completed" else f"{k.replace('_', ' ')}: {v if not isinstance(v, list) else ', '.join(v)}") for k, v in self.slots.items() if k in QUESTIONS)
        return (f"Thanks — noted ({got}). " if got else "Hi! I'm the NVU course advisor. I'll collect a few details, then suggest courses backed by the catalog and registration rules. ") + QUESTIONS[self.awaiting]

def demo_chat(messages, bot=None):
    bot, log = bot or AdvisorChatbot(), []
    for m in messages:
        a = bot.reply(m); log.append({"student": m, "advisor": a})
        print(f"\nStudent: {m}\nAdvisor: {a}")
    return log


### Chatbot demonstration (scripted conversation)

In [11]:
CHAT_SCRIPT = [
    "Hi, I need help registering for next semester.",
    "I'm in my second year, spring semester.",
    "My GPA is 3.1",
    "I completed CS101 (A), CS102 (A-), MATH101 (B+), ENG101 (B), PHYS101 (B), CS201 (B+). I also have 14 transfer credits.",
    "I'm interested in machine learning, data science and Python.",
    "My goal is a career in AI and data science. I'd like 15 credits and a medium workload.",
    "Why can't I take AI301?",
    "What is the maximum number of credits I can take?",
    "Actually I'd rather take only 12 credits.",
]
CHAT_LOG = demo_chat(CHAT_SCRIPT)



Student: Hi, I need help registering for next semester.
Advisor: Hi! I'm the NVU course advisor. I'll collect a few details, then suggest courses backed by the catalog and registration rules. Which academic year are you in (1–4)?

Student: I'm in my second year, spring semester.
Advisor: Thanks — noted (academic year: 2, semester: Spring). What is your current cumulative GPA? (type 'skip' if you don't have one yet)

Student: My GPA is 3.1
Advisor: Thanks — noted (academic year: 2, semester: Spring, gpa: 3.1). Which courses have you completed, with grades? e.g. “CS101 (A), CS102 (B+)” — or say “none” if you are a new student.

Student: I completed CS101 (A), CS102 (A-), MATH101 (B+), ENG101 (B), PHYS101 (B), CS201 (B+). I also have 14 transfer credits.
Advisor: Thanks — noted (academic year: 2, semester: Spring, gpa: 3.1, 6 completed courses). What are you interested in? e.g. “machine learning, web development”.

Student: I'm interested in machine learning, data science and Python.
Adv

### Interactive chat (optional)
Run the cell below and type messages (`quit` to stop). With `NVIDIA_API_KEY` (or OpenAI/Anthropic) set, an LLM extracts the profile details and answers free-form policy questions from the retrieved passages.


In [12]:
def chat_cli():
    bot = AdvisorChatbot(); print("NVU Course Advisor (type 'quit' to exit)\nAdvisor: " + bot.reply("hello"))
    while True:
        try: msg = input("\nYou: ").strip()
        except EOFError: break
        if msg.lower() in ("quit", "exit", "q"): break
        print("\nAdvisor: " + bot.reply(msg))

# chat_cli()    # <- uncomment to chat interactively


## 10. Save sample inputs and outputs (submission files)

In [13]:
import csv
(OUT_DIR / "all_profiles_report.txt").write_text("\n\n".join(format_result(r) for r in RESULTS.values()), encoding="utf-8")
for sid, r in RESULTS.items():
    (OUT_DIR / f"profile_{sid}_input.json").write_text(json.dumps(r["student"], indent=2, ensure_ascii=False), encoding="utf-8")
    (OUT_DIR / f"profile_{sid}_output.json").write_text(json.dumps({k: v for k, v in r.items() if k != "student"}, indent=2, ensure_ascii=False), encoding="utf-8")
    (OUT_DIR / f"profile_{sid}_report.txt").write_text(format_result(r), encoding="utf-8")
(OUT_DIR / "chatbot_transcript.json").write_text(json.dumps(CHAT_LOG, indent=2, ensure_ascii=False), encoding="utf-8")
with open(OUT_DIR / "summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["student_id", "name", "year", "semester", "gpa", "requested_credits", "credit_limit", "total_credits", "accepted", "flagged", "rule_audit"])
    for sid, r in RESULTS.items():
        st = r["student"]; w.writerow([sid, st["name"], st["academic_year"], st["semester"], st["gpa"], st["desired_credits"], r["credit_policy"]["limit"], r["total_credits"],
                                      " ".join(a["code"] for a in r["accepted"]), " ".join(e["code"] for e in r["excluded"] if e["status"] == "FLAGGED"), "PASS" if r["validation"]["passed"] else "FAIL"])
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

Saved: ['all_profiles_report.txt', 'chatbot_transcript.json', 'profile_S001_input.json', 'profile_S001_output.json', 'profile_S001_report.txt', 'profile_S002_input.json', 'profile_S002_output.json', 'profile_S002_report.txt', 'profile_S003_input.json', 'profile_S003_output.json', 'profile_S003_report.txt', 'profile_S004_input.json', 'profile_S004_output.json', 'profile_S004_report.txt', 'profile_S005_input.json', 'profile_S005_output.json', 'profile_S005_report.txt', 'profile_S006_input.json', 'profile_S006_output.json', 'profile_S006_report.txt', 'summary.csv']
